In [1]:
import pandas as pd
import numpy as np

In [7]:
data = pd.read_csv("multiple_linear_regression_dataset.csv")
print(data.head())
print("\n Column names:" ,data.columns)
print("\n Shape:",data.shape)


   age  experience  income
0   25           1   30450
1   30           3   35670
2   47           2   31580
3   32           5   40130
4   43          10   47830

 Column names: Index(['age', 'experience', 'income'], dtype='object')

 Shape: (20, 3)


In [10]:
X = data[["age", "experience"]].values
y = data["income"].values
print("Shape of X (inputs):", X.shape)  #20 samples, 2 features
print("Shape of y (output):", y.shape)  #20 income values

Shape of X (inputs): (20, 2)
Shape of y (output): (20,)


In [12]:
n_features = X.shape[1]
# Initialize weights to zeros
# We need ONE weight for EACH feature
# Think of weights as: "How much does each feature matter?"
# Starting at zero means the model has no opinion yet
w = np.zeros(n_features)
b = 0.0
w
b

0.0

In [13]:
def predict(X, w, b):
    """
    Calculate predicted income using the linear equation:
    y_hat = w1*x1 + w2*x2 + b

    In our case:
    predicted_income = (weight1 * age) + (weight2 * experience) + bias

    This is the same perceptron you know, but WITHOUT an activation function.
    We're just doing a weighted sum - no thresholding, no sigmoid, nothing.

    Parameters:
    X : input features (all samples)
    w : weights (one per feature)
    b : bias term

    Returns:
    y_hat : predicted values (predicted incomes)
    """
    # Matrix multiplication: for each row in X, compute weighted sum
    # X.dot(w) computes [x1*w1 + x2*w2] for each sample
    # Then we add bias b to every prediction
    y_hat = X.dot(w) + b

    return y_hat

In [16]:
initial_predictions = predict(X, w, b)


In [17]:
def mean_squared_error(y, y_hat):
    """
    Calculate how wrong our predictions are using Mean Squared Error (MSE).

    Formula: MSE = (1/N) * Σ(y_hat - y)²

    For each sample:
    1. Find the error: (predicted - actual)
    2. Square it: (error)²
    3. Average all squared errors

    Why MSE?
    - It penalizes large mistakes MORE than small ones (because of squaring)
    - If prediction is off by $10,000, error² = 100,000,000 (huge penalty!)
    - If prediction is off by $100, error² = 10,000 (small penalty)
    - This makes the model work HARD to fix big mistakes

    Parameters:
    y : actual income values
    y_hat : predicted income values

    Returns:
    loss : average squared error across all predictions
    """
    # Calculate error for each prediction, square it, then take the mean
    loss = ((y_hat - y) ** 2).mean()

    return loss

In [18]:
initial_loss = mean_squared_error(y, initial_predictions)

In [19]:
# - We square the error to make all errors positive (otherwise +100 and -100 would cancel out)
# - Squaring AMPLIFIES large mistakes, forcing the model to prioritize fixing them
# - If one prediction is very wrong (off by $10,000), the loss skyrockets
# - We don't just use absolute error because:
#   * Squared error is differentiable (smooth gradient)
#   * It penalizes outliers more heavily
#   * Mathematically easier to work with in gradient descent

In [22]:
def compute_gradients(X, y, y_hat):
    """
    Calculate how to adjust weights and bias to reduce the loss.

    This is the CORE of learning!
    Gradients tell us:
    - Which direction to move (increase or decrease weights)
    - How much to move (magnitude of change)

    Mathematical formulas (from calculus):
    - Gradient w.r.t. weights: dw = (2/N) * X^T * (y_hat - y)
    - Gradient w.r.t. bias:    db = (2/N) * Σ(y_hat - y)

    Intuition:
    - If error is positive (predicted too high), gradients push weights DOWN
    - If error is negative (predicted too low), gradients push weights UP
    - The bigger the error, the bigger the push

    Parameters:
    X : input features
    y : actual values
    y_hat : predicted values

    Returns:
    dw : gradient for weights (how much to change each weight)
    db : gradient for bias (how much to change bias)
    """
    # Number of samples
    N = len(y)

    # Calculate error: difference between predictions and actual values
    error = y_hat - y

    # Gradient for weights:
    # X.T (transpose) is needed for matrix multiplication
    # This computes how much each feature contributed to the error
    dw = (2 / N) * X.T.dot(error)

    # Gradient for bias:
    # Just the average error (bias affects all predictions equally)
    db = (2 / N) * error.sum()

    return dw, db

# Test gradient calculation
dw_test, db_test = compute_gradients(X, y, initial_predictions)
print("Gradient for weights:", dw_test)
print("Gradient for bias:", db_test)


Gradient for weights: [-3315904.  -570214.]
Gradient for bias: -81471.0


In [23]:
def update_parameters(w, b, dw, db, lr):
    """
    Adjust weights and bias to reduce the loss.

    Update rule:
    - new_weight = old_weight - (learning_rate * gradient)
    - new_bias = old_bias - (learning_rate * gradient)

    The learning rate (lr) controls the step size:
    - Too large → model bounces around, never settles, might explode
    - Too small → learning takes FOREVER, might get stuck
    - Just right → smooth, steady improvement

    Why subtract?
    - Gradients point in the direction of INCREASING loss
    - We want to DECREASE loss, so we go in the opposite direction
    - Minus sign = "move away from higher loss"

    Parameters:
    w : current weights
    b : current bias
    dw : gradient for weights
    db : gradient for bias
    lr : learning rate (step size)

    Returns:
    w : updated weights
    b : updated bias
    """
    # Move weights in the opposite direction of the gradient
    w = w - lr * dw

    # Move bias in the opposite direction of its gradient
    b = b - lr * db

    return w, b

# Example: update once with learning rate 0.0001
w_updated, b_updated = update_parameters(w, b, dw_test, db_test, lr=0.0001)
print("Weights after one update:", w_updated)
print("Bias after one update:", b_updated)

Weights after one update: [331.5904  57.0214]
Bias after one update: 8.1471


In [24]:
# training loop
w = np.zeros(n_features)
b = 0.0
lr = 0.0001
epochs = 1000

for epoch in range(epochs):
    # 1. FORWARD PASS: Make predictions with current weights
    y_hat = predict(X, w, b)

    # 2. CALCULATE LOSS: See how wrong we are
    loss = mean_squared_error(y, y_hat)

    # 3. COMPUTE GRADIENTS: Figure out how to improve
    dw, db = compute_gradients(X, y, y_hat)

    # 4. UPDATE PARAMETERS: Actually improve the weights
    w, b = update_parameters(w, b, dw, db, lr)

    # Print progress every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss:,.2f}")

print("-" * 50)
print("Training Complete!")
print(f"\nFinal weights: {w}")
print(f"Final bias: {b:.2f}")
print(f"Final loss: {loss:,.2f}")

Epoch    0 | Loss: 1,727,049,635.00
Epoch  100 | Loss: 66,491,868.55
Epoch  200 | Loss: 61,752,567.20
Epoch  300 | Loss: 58,616,531.08
Epoch  400 | Loss: 56,528,801.54
Epoch  500 | Loss: 55,126,542.03
Epoch  600 | Loss: 54,172,526.95
Epoch  700 | Loss: 53,511,656.14
Epoch  800 | Loss: 53,042,523.73
Epoch  900 | Loss: 52,698,829.56
--------------------------------------------------
Training Complete!

Final weights: [ 764.75405919 1371.03430441]
Final bias: 321.74
Final loss: 52,439,538.18


In [25]:
print("\n" + "=" * 60)
print("FINAL MODEL PARAMETERS")
print("=" * 60)
print(f"Final weights: {w}")
print(f"Final bias: {b:.2f}")
print(f"Final loss: {loss:,.2f}")
print("=" * 60)

# Comparison of Actual vs Predicted on training data
print("\nComparison of Actual vs Predicted (first 10 samples):")
print("-" * 60)
final_predictions = predict(X, w, b)
for i in range(10):
    print(f"Sample {i+1}: Actual = ${y[i]:,.2f} | Predicted = ${final_predictions[i]:,.2f} | Error = ${abs(y[i] - final_predictions[i]):,.2f}")

# PREDICTION FOR NEW CANDIDATE
print("\n" + "=" * 60)
print("PREDICTION FOR NEW CANDIDATE")
print("=" * 60)

# Example: Someone who is 35 years old with 7 years of experience
new_candidate = np.array([35, 7])
predicted_salary = new_candidate.dot(w) + b

print(f"New Candidate Profile:")
print(f"  - Age: {new_candidate[0]} years")
print(f"  - Experience: {new_candidate[1]} years")
print(f"\nPredicted Income: ${predicted_salary:,.2f}")
print("=" * 60)


FINAL MODEL PARAMETERS
Final weights: [ 764.75405919 1371.03430441]
Final bias: 321.74
Final loss: 52,439,538.18

Comparison of Actual vs Predicted (first 10 samples):
------------------------------------------------------------
Sample 1: Actual = $30,450.00 | Predicted = $20,811.62 | Error = $9,638.38
Sample 2: Actual = $35,670.00 | Predicted = $27,377.46 | Error = $8,292.54
Sample 3: Actual = $31,580.00 | Predicted = $39,007.25 | Error = $7,427.25
Sample 4: Actual = $40,130.00 | Predicted = $31,649.04 | Error = $8,480.96
Sample 5: Actual = $47,830.00 | Predicted = $46,916.50 | Error = $913.50
Sample 6: Actual = $41,630.00 | Predicted = $48,921.43 | Error = $7,291.43
Sample 7: Actual = $41,340.00 | Predicted = $28,590.02 | Error = $12,749.98
Sample 8: Actual = $37,650.00 | Predicted = $31,042.76 | Error = $6,607.24
Sample 9: Actual = $40,250.00 | Predicted = $35,472.81 | Error = $4,777.19
Sample 10: Actual = $45,150.00 | Predicted = $41,115.42 | Error = $4,034.58

PREDICTION FOR NEW 